# VQA High Performance — 0.9+ 목표
- 모델: `Qwen/Qwen2.5-VL-7B-Instruct` (7B)
- 전략: 전체 데이터 + 5epoch + LoRA r16 + CoT 프롬프트 + 앙상블

## 0. GPU 확인

In [ ]:
!nvidia-smi

## 1. 패키지 설치 (최초 1회)

In [ ]:
# # 최초 1회만 실행
# !pip install transformers accelerate peft bitsandbytes datasets pillow qwen-vl-utils
# !pip uninstall torch torchvision torchaudio -y
# !pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128

## 2. 라이브러리 로드

In [ ]:
import os
import math
import random
import numpy as np
import pandas as pd
from pathlib import Path
from dataclasses import dataclass
from typing import Any
from PIL import Image
from tqdm.auto import tqdm
from collections import Counter

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 3. Config

In [ ]:
# ── 경로 ──
BASE_DIR  = Path('.')           # 데이터셋 루트 (본인 환경에 맞게 수정)
TRAIN_CSV = BASE_DIR / 'train.csv'
TEST_CSV  = BASE_DIR / 'test.csv'
SAVE_DIR  = BASE_DIR / 'best_model'

# ── 모델: 3B → 7B 업그레이드 ──
MODEL_ID   = 'Qwen/Qwen2.5-VL-7B-Instruct'
IMAGE_SIZE = 336

# ── 학습 ──
TRAIN_SAMPLE_N = None   # 전체 데이터 사용
VAL_RATIO      = 0.1
BATCH_SIZE     = 1
GRAD_ACCUM     = 8      # 유효 배치 8
EPOCHS         = 5
LR             = 2e-4

# ── LoRA: rank 8 → 16 ──
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

# ── 앙상블: 시드 여러 개 ──
ENSEMBLE_SEEDS = [42, 123, 777]   # 각 시드로 학습 후 voting

LABEL2ID = {'a': 0, 'b': 1, 'c': 2, 'd': 3}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

## 4. 데이터 로드

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

# 결측치 제거
train_df = train_df.dropna(subset=['answer'])
train_df = train_df[train_df['answer'].isin(['a','b','c','d'])].reset_index(drop=True)

# 누락 이미지 필터링
train_df = train_df[train_df['path'].apply(lambda p: (BASE_DIR / p).exists())].reset_index(drop=True)
print(f'필터링 후 train: {len(train_df)} | test: {len(test_df)}')
print(f'정답 분포:\n{train_df["answer"].value_counts()}')

## 5. 프롬프트 설계 (CoT 강화)

In [ ]:
# 3단계: CoT 유도 프롬프트
SYSTEM_INSTRUCT = (
    "You are an expert at analyzing images and answering multiple choice questions about recycling and materials. "
    "Look carefully at the image, identify the objects and their materials, "
    "then select the single best answer. "
    "Answer using exactly one letter: a, b, c, or d. No explanation."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
    )

row = train_df.iloc[0]
print(build_mc_prompt(row['question'], row['a'], row['b'], row['c'], row['d']))

## 6. Dataset & Collator

In [ ]:
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, base_dir, is_train=True):
        self.df        = df.reset_index(drop=True)
        self.processor = processor
        self.base_dir  = Path(base_dir)
        self.is_train  = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(self.base_dir / row['path']).convert('RGB')
        user_text = build_mc_prompt(
            str(row['question']),
            str(row['a']), str(row['b']),
            str(row['c']), str(row['d'])
        )
        messages = [
            {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_INSTRUCT}]},
            {'role': 'user',   'content': [
                {'type': 'image', 'image': img},
                {'type': 'text',  'text': user_text}
            ]}
        ]
        if self.is_train:
            gold = str(row['answer']).strip().lower()
            messages.append({'role': 'assistant', 'content': [{'type': 'text', 'text': gold}]})

        return {'messages': messages, 'image': img,
                'answer': str(row['answer']) if self.is_train else None}


@dataclass
class DataCollator:
    processor: Any
    is_train: bool = True

    def __call__(self, batch):
        texts, images = [], []
        for sample in batch:
            text = self.processor.apply_chat_template(
                sample['messages'], tokenize=False, add_generation_prompt=False
            )
            texts.append(text)
            images.append(sample['image'])

        enc = self.processor(
            text=texts, images=images, padding=True, return_tensors='pt'
        )
        if self.is_train:
            enc['labels'] = enc['input_ids'].clone()
        return enc

## 7. 모델 로드 함수

In [ ]:
def load_model(seed=42):
    torch.manual_seed(seed)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        min_pixels=IMAGE_SIZE * IMAGE_SIZE,
        max_pixels=IMAGE_SIZE * IMAGE_SIZE,
        trust_remote_code=True
    )
    base_model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True
    )
    base_model = prepare_model_for_kbit_training(base_model)
    base_model.gradient_checkpointing_enable()

    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias='none',
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        task_type='CAUSAL_LM'
    )
    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()
    return model, processor

## 8. 학습 & 검증 함수

In [ ]:
def extract_choice(text: str) -> str:
    text = text.strip().lower()
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines: return 'a'
    last = lines[-1]
    if last in ['a','b','c','d']: return last
    for tok in last.split():
        if tok in ['a','b','c','d']: return tok
    return 'a'


def train_one_epoch(model, loader, optimizer, scheduler, scaler, epoch):
    model.train()
    running, avg_loss = 0.0, 0.0
    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(loader, desc=f'Epoch {epoch} [train]', unit='batch')
    for step, batch in enumerate(pbar, start=1):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            loss = model(**batch).loss / GRAD_ACCUM
        scaler.scale(loss).backward()
        running += loss.item()

        if step % GRAD_ACCUM == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            avg_loss = running / GRAD_ACCUM
            pbar.set_postfix({'loss': f'{avg_loss:.3f}', 'lr': f'{scheduler.get_last_lr()[0]:.2e}'})
            running = 0.0
    return avg_loss


@torch.no_grad()
def evaluate(model, loader, processor):
    model.eval()
    val_loss, correct, total = 0.0, 0, 0

    choice_ids = {
        c: processor.tokenizer.encode(c, add_special_tokens=False)[0]
        for c in ['a','b','c','d']
    }
    choice_id_set = set(choice_ids.values())

    pbar = tqdm(loader, desc='[valid]', unit='batch')
    for batch in pbar:
        batch  = {k: v.to(DEVICE) for k, v in batch.items()}
        labels = batch.pop('labels')

        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            outputs = model(**batch)

        for i in range(labels.size(0)):
            label_seq = labels[i].tolist()
            gold_pos  = None
            for pos in reversed(range(len(label_seq))):
                if label_seq[pos] in choice_id_set:
                    gold_pos = pos
                    break
            if gold_pos is None or gold_pos == 0:
                total += 1
                continue

            logit_pos = gold_pos - 1
            scores    = {c: outputs.logits[i][logit_pos][tid].item() for c, tid in choice_ids.items()}
            pred      = max(scores, key=scores.get)
            gold      = next(c for c, tid in choice_ids.items() if tid == label_seq[gold_pos])

            if pred == gold: correct += 1
            total += 1

    acc = correct / total if total > 0 else 0
    return val_loss / len(loader), acc

## 9. 학습 루프 (시드별 반복 — 앙상블용)

In [ ]:
# 앙상블 시드 중 첫 번째로 단일 학습 먼저 진행
# 전체 앙상블은 아래 셀에서 수행
CURRENT_SEED = ENSEMBLE_SEEDS[0]

random.seed(CURRENT_SEED)
np.random.seed(CURRENT_SEED)
torch.manual_seed(CURRENT_SEED)

# Train / Val 분리
val_size  = int(len(train_df) * VAL_RATIO)
val_df    = train_df.sample(val_size, random_state=CURRENT_SEED)
train_df2 = train_df.drop(val_df.index).reset_index(drop=True)
val_df    = val_df.reset_index(drop=True)
print(f'Train: {len(train_df2)} | Val: {len(val_df)} | Test: {len(test_df)}')

# 모델 로드
model, processor = load_model(seed=CURRENT_SEED)

# DataLoader
train_ds = VQAMCDataset(train_df2, processor, BASE_DIR, is_train=True)
val_ds   = VQAMCDataset(val_df,    processor, BASE_DIR, is_train=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=DataCollator(processor, True),  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=DataCollator(processor, True),  num_workers=0)

# Optimizer & Scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
num_training_steps = EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)
num_warmup_steps   = int(num_training_steps * 0.05)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=num_training_steps
)
scaler = torch.amp.GradScaler('cuda', enabled=True)

print(f'총 학습 스텝: {num_training_steps} | 웜업: {num_warmup_steps}')

In [ ]:
best_val_acc = 0
history = []
save_path = SAVE_DIR / f'seed_{CURRENT_SEED}'

for epoch in range(1, EPOCHS + 1):
    print(f'\n[Epoch {epoch}/{EPOCHS}] seed={CURRENT_SEED}')
    train_loss        = train_one_epoch(model, train_loader, optimizer, scheduler, scaler, epoch)
    val_loss, val_acc = evaluate(model, val_loader, processor)

    history.append({'epoch': epoch, 'train_loss': train_loss,
                    'val_loss': val_loss, 'val_acc': val_acc})

    print(f'  Train Loss : {train_loss:.4f}')
    print(f'  Val   Loss : {val_loss:.4f} | Val Acc : {val_acc:.4f}')
    print(f'  LR         : {scheduler.get_last_lr()[0]:.2e}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        model.save_pretrained(save_path)
        processor.save_pretrained(save_path)
        print(f'  ✅ Best 모델 저장 → {save_path} (Val Acc: {best_val_acc:.4f})')

print(f'\n[seed {CURRENT_SEED}] 최종 Best Val Accuracy: {best_val_acc:.4f}')

## 10. 학습 곡선 시각화

In [ ]:
import matplotlib.pyplot as plt

hist_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(hist_df['epoch'], hist_df['train_loss'], label='Train Loss')
axes[0].plot(hist_df['epoch'], hist_df['val_loss'],   label='Val Loss')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(hist_df['epoch'], hist_df['val_acc'], label='Val Acc', color='green', marker='o')
axes[1].set_title('Val Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylim(0, 1)
axes[1].legend()

plt.tight_layout()
plt.show()

## 11. 단일 모델 추론

In [ ]:
@torch.no_grad()
def predict_single(model, df, processor, base_dir):
    """로짓 직접 비교 방식 — generate() 대비 빠름"""
    model.eval()
    preds = []

    choice_ids = {
        c: processor.tokenizer.encode(c, add_special_tokens=False)[0]
        for c in ['a','b','c','d']
    }

    for i in tqdm(range(len(df)), desc='Inference', unit='sample'):
        row = df.iloc[i]
        img = Image.open(Path(base_dir) / row['path']).convert('RGB')
        user_text = build_mc_prompt(
            str(row['question']),
            str(row['a']), str(row['b']),
            str(row['c']), str(row['d'])
        )
        messages = [
            {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_INSTRUCT}]},
            {'role': 'user',   'content': [
                {'type': 'image', 'image': img},
                {'type': 'text',  'text': user_text}
            ]}
        ]
        text   = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[img], return_tensors='pt').to(DEVICE)

        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            outputs = model(**inputs)

        last_logits = outputs.logits[0, -1, :]
        scores = {c: last_logits[tid].item() for c, tid in choice_ids.items()}
        preds.append(max(scores, key=scores.get))

    return preds


# 단일 모델 추론
single_preds = predict_single(model, test_df, processor, BASE_DIR)
print(pd.Series(single_preds).value_counts())

## 12. 앙상블 추론 (다중 시드 voting)

In [ ]:
# ── 앙상블: 저장된 모델들 불러와서 voting ──
# 각 시드로 학습된 모델이 SAVE_DIR/seed_{N} 에 저장되어 있어야 함

def load_trained_model(seed):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    save_path = SAVE_DIR / f'seed_{seed}'
    processor = AutoProcessor.from_pretrained(save_path, trust_remote_code=True)
    base_model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, device_map='auto', trust_remote_code=True
    )
    model = PeftModel.from_pretrained(base_model, save_path)
    return model, processor


all_preds = []   # 각 모델의 예측 리스트

for seed in ENSEMBLE_SEEDS:
    save_path = SAVE_DIR / f'seed_{seed}'
    if not save_path.exists():
        print(f'seed {seed} 모델 없음 → 스킵 (학습 후 실행하세요)')
        continue

    print(f'\n[seed {seed}] 모델 로드 중...')
    m, p = load_trained_model(seed)
    preds = predict_single(m, test_df, p, BASE_DIR)
    all_preds.append(preds)
    print(f'[seed {seed}] 예측 완료: {pd.Series(preds).value_counts().to_dict()}')

    # 메모리 해제
    del m, p
    torch.cuda.empty_cache()

print(f'\n앙상블 모델 수: {len(all_preds)}')

In [ ]:
# Majority Voting
if len(all_preds) > 0:
    final_preds = []
    for i in range(len(test_df)):
        votes = [preds[i] for preds in all_preds]
        final_preds.append(Counter(votes).most_common(1)[0][0])
else:
    # 앙상블 모델 없으면 단일 모델 예측 사용
    print('앙상블 모델 없음 → 단일 모델 예측 사용')
    final_preds = single_preds

print(f'최종 예측 분포:\n{pd.Series(final_preds).value_counts()}')

## 13. 제출 파일 생성

In [ ]:
submission = pd.DataFrame({'id': test_df['id'], 'answer': final_preds})
submission.to_csv('submission.csv', index=False)

print('submission.csv 저장 완료')
print(f'제출 파일 크기: {len(submission)}')
print(submission['answer'].value_counts())
submission.head(10)